In [ ]:
#| default_exp machine_learning.training.token_classification

> **AI-generated — pending human review.** Created by Claude Code on 2026-10-22. Review checklist: `_nbdev_development/AI_REVIEW_CHECKLIST.md` (workspace root). Remove this cell only after review passes.

# Token classification training (zero-width seams)

Consumes the row contract of `machine_learning.producers.note_division`: each row is one text with zero-width `{start, end, label: 'split'}` spans. Nothing here knows about vaults, datasets on disk, or machines — data paths and devices are caller-supplied, so this module is safe to publish.

**Label rule.** Byte-level BPE means token boundaries do not line up with words, so labels are snapped by *character offsets*: the last non-blank token whose range **ends at** a seam gets label 1; a whitespace-only token covering the seam (the `\n` joiner in canonical rows) gets `-100` (ignored); everything else, including padding and special tokens, is ignored or 0. Train on the newline-stripped view (`ml_augmentation.division_strip_newlines_v1`) so the seam carries no input-visible cue. Truncation cuts spans past `max_length` out of the window — feed over-cap rows through `chunk_division_rows` first (see "Chunking over-cap rows") so no seam is lost and no tail is wasted.

In [ ]:
#| export
from bisect import bisect_left
from os import PathLike
from pathlib import Path

import contextlib
import os
import time
import torch
from transformers import (AutoModelForTokenClassification,
                                      AutoTokenizer)

## Labels: zero-width seams → per-token `0/1/-100`

In [ ]:
#| export
def token_labels_for_spans(text: str, spans: list[dict], offsets: list[tuple]) -> list[int]:
    """
    Per-token `0/1/-100` labels: 1 on the last non-blank token ending exactly at a seam.

    `offsets` are the tokenizer's character `offset_mapping` for the same text
    (special tokens excluded, i.e. `add_special_tokens=False`). A whitespace-only
    token *starting at* a seam (the canonical `\n` joiner) is `-100` — invisible
    to the loss; in the newline-stripped view no such token exists and the seam
    is an ordinary space. Empty ranges (special tokens) never match.

    **Parameters**
    - `text` (*str*) — The row text.
    - `spans` (*list[dict]*) — Zero-width spans with `label == 'split'`.
    - `offsets` (*list[tuple]*) — `(start, end)` char range per token.
    """
    seams = {s['start'] for s in spans if s.get('label') == 'split'}
    labels = []
    for b, e in offsets:
        if e > b and e in seams and text[b:e].strip():
            labels.append(1)
        elif b in seams and not text[b:e].strip():
            labels.append(-100)
        else:
            labels.append(0)
    return labels


In [ ]:
from fastcore.test import test_eq

# canonical view: 'ab' ends at the seam -> 1; the '\n' token covering it -> -100.
offs = [(0, 2), (2, 3), (3, 5)]
seams = [{'start': 2, 'end': 2, 'label': 'split'}]
test_eq(token_labels_for_spans('ab\ncd', seams, offs), [1, -100, 0])
# stripped view: seam glued to the next token -> no -100 anywhere, just 1 then 0.
test_eq(token_labels_for_spans('ab cd', seams, [(0, 2), (2, 5)]), [1, 0])
# negatives: no spans -> all zero; blank token touching a seam start is not positive.
test_eq(token_labels_for_spans('zz yy', [], [(0, 2), (2, 3), (3, 5)]), [0, 0, 0])


## Features and scoring

In [ ]:
#| export
def division_feature_tensors(rows: list[dict], tokenizer, max_length: int = 512) -> dict:
    """
    Rows → padded `input_ids`/`attention_mask`/`labels` tensors (`-100` pad on labels).

    No special tokens (offsets stay aligned with plain text). Truncation past
    `max_length` drops seams whose token vanished — the row silently becomes all-
    negative; chunk over-cap rows with `chunk_division_rows` beforehand so no
    seam is lost, and check `(labels == 1).sum()` against your seam count when
    changing `max_length`.

    **Parameters**
    - `rows` (*list[dict]*) — Rows with `text` and zero-width `spans`.
    - `tokenizer` — A fast tokenizer (needs `offset_mapping`).
    - `max_length` (*int*) — Truncation/padding width cap.
    """
    texts = [r['text'] for r in rows]
    enc = tokenizer(texts, return_offsets_mapping=True, add_special_tokens=False,
                    truncation=True, max_length=max_length, padding="longest")
    labels = [token_labels_for_spans(t, r['spans'], o)
              for t, r, o in zip(texts, rows, enc.pop('offset_mapping'))]
    width = len(enc['input_ids'][0])
    return {'input_ids': torch.tensor(enc['input_ids']),
            'attention_mask': torch.tensor(enc['attention_mask']),
            'labels': torch.tensor([l + [-100] * (width - len(l)) for l in labels])}

In [ ]:
#| export
def predict_seam_positions(model, tokenizer, texts: list[str], max_length: int = 512,
                           batch_size: int = 8) -> list[list[int]]:
    """
    Predicted seam char positions per text: end offsets of all label-1 tokens.

    The inference counterpart of `score_seam_f1`'s matching rule — the deployment
    entry point for note division. Feed the newline-stripped view
    (`ml_augmentation.strip_newlines_text`) so inputs match what training saw;
    returned positions are char offsets into the text you passed. Runs on the
    model's current device and leaves it in eval mode.

    **Parameters**
    - `model` — A loaded `AutoModelForTokenClassification` (`num_labels=2`).
    - `tokenizer` — The model's tokenizer.
    - `texts` (*list[str]*) — Texts to predict on (in the training view).
    - `max_length` (*int*) — Window cap; must match training or seams truncate away.
    - `batch_size` (*int*) — Inference batch size.
    """
    model.eval()
    out = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(texts[i:i + batch_size], add_special_tokens=False, truncation=True,
                        max_length=max_length, padding='longest',
                        return_offsets_mapping=True, return_tensors='pt')
        offs = enc.pop('offset_mapping').tolist()
        with torch.no_grad(), _autocast(model.device):
            preds = model(input_ids=enc['input_ids'].to(model.device),
                          attention_mask=enc['attention_mask'].to(model.device)
                          ).logits.argmax(-1).tolist()
        out += [sorted({o[k][1] for k in range(len(o)) if pr[k] == 1 and o[k][1] > o[k][0]})
                for o, pr in zip(offs, preds)]
    return out


def score_seam_f1(model, tokenizer, rows: list[dict], max_length: int = 512,
                  batch_size: int = 8) -> dict:
    """
    Seam-level precision/recall/F1: a hit is a predicted position equal to a gold seam's `start`.

    Exact-boundary matching (zero-width spans), micro-averaged over rows — the
    gate metric for the division task. Predictions come from
    `predict_seam_positions`, so scoring and deployment share one matching rule.
    Runs on the model's current device.

    **Parameters**
    - `model` — A loaded `AutoModelForTokenClassification` (`num_labels=2`).
    - `tokenizer` — The model's tokenizer.
    - `rows` (*list[dict]*) — Scored rows (pure eval rows by convention).
    - `max_length` (*int*) — Must match training or seams truncate away.
    - `batch_size` (*int*) — Inference batch size.
    """
    tp = fp = fn = 0
    preds = predict_seam_positions(model, tokenizer, [r['text'] for r in rows],
                                   max_length=max_length, batch_size=batch_size)
    for r, pred in zip(rows, preds):
        gold = {s['start'] for s in r['spans'] if s['label'] == 'split'}
        p = set(pred)
        tp += len(p & gold); fp += len(p - gold); fn += len(gold - p)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rc = tp / (tp + fn) if tp + fn else 0.0
    return {'precision': prec, 'recall': rc, 'f1': 2 * prec * rc / (prec + rc) if prec + rc else 0.0}


## Chunking over-cap rows (no wasted tails)

`division_feature_tensors` truncates silently: a seam past `max_length` vanishes and the row becomes an all-negative false label. Dropping such rows instead wastes their content — long atomic notes are exactly the negatives we want, and the tail of a positive row can hold other seams. The fix is sliding-window chunking applied to *rows* before batching: over-cap rows become several ≤`max_length`-token sub-rows sharing `overlap` tokens. Each gold seam is kept in **exactly one** window (the covering one with the most balanced context), so positives are neither lost nor double-counted, and artificial cut points carry no label — they are ordinary negatives. The same windows serve inference (`predict_seam_positions_windowed`), so train and deploy views match.

In [ ]:
#| export
def _seam_homes(spans: list[dict], wins: list[tuple[int, int]], ends: list[int]) -> list[int]:
    """Per-span window index: the covering window (both neighbor tokens inside) whose center is nearest.

    Center-nearest assignment gives each seam the most balanced left/right
    context and keeps every gold seam in exactly one window — chunking neither
    loses nor double-counts a positive. Assumes seams sit at token boundaries of
    the row (`\\n`-joined division rows satisfy this after newline stripping).
    """
    mids = [(x + y) / 2 for x, y in wins]
    homes = []
    for s in spans:
        k = bisect_left(ends, s['start'])
        lo = max(k - 1, 0)
        homes.append(min((j for j, (x, y) in enumerate(wins) if x <= lo and y > k),
                         key=lambda j: abs(mids[j] - k)))
    return homes


def _chunk_division_row_offsets(row: dict, offs: list[tuple], max_length: int,
                                overlap: int) -> list[tuple[int, dict]]:
    """Pre-offset version of `division_row_windows` (shared by the batched fast path)."""
    if len(offs) <= max_length: return [(0, row)]
    if overlap >= max_length: raise ValueError('overlap must be < max_length')
    wins, a = [], 0
    while True:
        b = min(a + max_length, len(offs))
        wins.append((a, b))
        if b == len(offs): break
        a = b - overlap
    ends = [e for _, e in offs]
    homes = _seam_homes(row['spans'], wins, ends)
    by_win = {j: [s for i, s in enumerate(row['spans']) if homes[i] == j]
              for j in range(len(wins))}
    out = []
    for j, (x, y) in enumerate(wins):
        a0 = offs[x][0]                                   # last window keeps trailing whitespace too
        b0 = offs[y - 1][1] if j + 1 < len(wins) else len(row['text'])
        out.append((a0, {**row, 'text': row['text'][a0:b0],
                         'spans': [{'start': s['start'] - a0, 'end': s['end'] - a0,
                                    'label': s['label']} for s in by_win[j]]}))
    return out


def division_row_windows(row: dict, tokenizer, max_length: int = 512,
                         overlap: int = 64) -> list[tuple[int, dict]]:
    """
    Row → `(char_start, sub-row)` sliding windows of ≤`max_length` tokens; nothing wasted.

    Rows that fit are returned unchanged (offset 0). Over-cap rows — seam-free
    ones included, since their tails are real negatives — are cut into token
    windows sharing `overlap` tokens, with each gold seam kept in exactly one
    window and shifted to window coordinates; artificial cut points carry no
    label. Apply to the newline-stripped training view so windows align with
    what `division_feature_tensors` re-tokenizes.

    **Parameters**
    - `row` (*dict*) — Row with `text` and zero-width `spans`.
    - `tokenizer` — A fast tokenizer (needs `offset_mapping`).
    - `max_length` (*int*) — Token cap per window.
    - `overlap` (*int*) — Tokens shared between consecutive windows (< `max_length`).
    """
    offs = tokenizer(row['text'], add_special_tokens=False,
                     return_offsets_mapping=True)['offset_mapping']
    return _chunk_division_row_offsets(row, offs, max_length, overlap)


def chunk_division_rows(rows: list[dict], tokenizer, max_length: int = 512,
                        overlap: int = 64, batch_size: int = 256) -> list[dict]:
    """
    All rows through `division_row_windows`, tokenized in batches (dataset-scale fast path).

    Returns only the sub-rows (no char offsets), preserving per-row window order.
    This is the training-time entry point: feed it the stripped view right before
    `train_token_classifier` so no row ever truncates.

    **Parameters**
    - `rows` (*list[dict]*) — Rows to fit into `max_length`.
    - `tokenizer` — A fast tokenizer (needs `offset_mapping`).
    - `max_length` (*int*) / `overlap` (*int*) — As in `division_row_windows`.
    - `batch_size` (*int*) — Tokenization batch size.
    """
    out = []
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i + batch_size]
        encs = tokenizer([r['text'] for r in batch], add_special_tokens=False,
                         return_offsets_mapping=True)['offset_mapping']
        for row, offs in zip(batch, encs):
            out += [w for _, w in _chunk_division_row_offsets(row, offs, max_length, overlap)]
    return out

In [ ]:
# division_row_windows / chunk_division_rows: cap respected, seams kept exactly once,
# and every kept seam still produces a label-1 after re-tokenizing its window.
from fastcore.test import ExceptionExpected, test_eq
from transformers import AutoTokenizer

_tk = AutoTokenizer.from_pretrained('hf-internal-testing/tiny-random-BertForTokenClassification')
_unit = 'note one is here '                                   # 'here' ends 1 char before each unit boundary
_long = _unit * 40
_seams = [{'start': i, 'end': i, 'label': 'split'} for i in range(len(_unit) - 1, len(_long) - 1, len(_unit))]
_lrow = {'text': _long, 'spans': _seams}

# a row that fits passes through unchanged at offset 0.
test_eq(division_row_windows({'text': 'ab cd', 'spans': []}, _tk, max_length=64),
        [(0, {'text': 'ab cd', 'spans': []})])
# seam-free over-cap row is still chunked (the tail is a real negative, not waste).
assert len(division_row_windows({'text': _long, 'spans': []}, _tk, max_length=16, overlap=4)) > 1

wins = division_row_windows(_lrow, _tk, max_length=16, overlap=4)
# every window respects the token cap.
assert all(len(_tk(w['text'], add_special_tokens=False)['input_ids']) <= 16 for _, w in wins)
# windows cover the full char range and overlap (no wasted chars between cuts).
test_eq(wins[0][0], 0)
test_eq(wins[-1][0] + len(wins[-1][1]['text']), len(_long))
assert all(wins[j + 1][0] < wins[j][0] + len(wins[j][1]['text']) for j in range(len(wins) - 1))
# each gold seam appears exactly once across all windows, mapped back correctly.
back = sorted(a0 + s['start'] for a0, w in wins for s in w['spans'])
test_eq(back, [s['start'] for s in _seams])
# integration guarantee: re-tokenizing a window still labels each of its seams 1.
for _, w in wins:
    o = _tk(w['text'], add_special_tokens=False, return_offsets_mapping=True)['offset_mapping']
    labs = token_labels_for_spans(w['text'], w['spans'], o)
    test_eq(sum(l == 1 for l in labs), len(w['spans']))
# batched fast path == per-row slow path.
_rows = [{'text': 'ab cd', 'spans': []}, _lrow, {'text': _long, 'spans': []}]
test_eq(chunk_division_rows(_rows, _tk, max_length=16, overlap=4),
        [w for r in _rows for _, w in division_row_windows(r, _tk, max_length=16, overlap=4)])
# overlap >= max_length would loop forever — refused.
with ExceptionExpected(ex=ValueError, regex='overlap'):
    division_row_windows(_lrow, _tk, max_length=8, overlap=8)

In [ ]:
# Example: rescuing an over-cap row (tiny tokenizer, small cap for legibility).
from transformers import AutoTokenizer

_etok = AutoTokenizer.from_pretrained('hf-internal-testing/tiny-random-BertForTokenClassification')
_parts = [n.rstrip() for n in ['alpha beta gamma delta ' * 10, 'epsilon zeta eta ' * 10,
                               'theta iota kappa lambda ' * 8]]
_text, _spans, _pos = '', [], 0
for i, p in enumerate(_parts):                              # seam sits where the left token ends
    if i:
        _spans.append({'start': _pos, 'end': _pos, 'label': 'split'})
        _text += ' '
    _text += p
    _pos = len(_text)
print('tokens:', len(_etok(_text, add_special_tokens=False)['input_ids']), 'seams:', len(_spans))
_rows = chunk_division_rows([{'text': _text, 'spans': _spans}], _etok, max_length=32, overlap=4)
for r in _rows:
    print(f'window {len(r["text"]):4d} chars, seams at {[s["start"] for s in r["spans"]]}')
assert sum(len(r['spans']) for r in _rows) == len(_spans)   # every seam rescued exactly once

In [ ]:
#| export
def predict_seam_positions_windowed(model, tokenizer, texts: list[str], max_length: int = 512,
                                    overlap: int = 64, batch_size: int = 8) -> list[list[int]]:
    """
    `predict_seam_positions` with no truncation loss: over-cap texts run as sliding windows.

    The deployment counterpart of `chunk_division_rows` — train and infer on the
    same ≤`max_length` views, so a seam past the old cut cannot silently vanish at
    inference time either. Window predictions are mapped back to char offsets in
    the full text; a position seen in two overlapping windows is reported once.

    **Parameters**
    - `model` — A loaded `AutoModelForTokenClassification` (`num_labels=2`).
    - `tokenizer` — The model's tokenizer.
    - `texts` (*list[str]*) — Texts to predict on (in the training view).
    - `max_length` (*int*) / `overlap` (*int*) — Window geometry, matching training.
    - `batch_size` (*int*) — Inference batch size within a text's windows.
    """
    model.eval()
    out = []
    for t in texts:
        pos = set()
        for a0, w in division_row_windows({'text': t, 'spans': []}, tokenizer, max_length, overlap):
            ps = predict_seam_positions(model, tokenizer, [w['text']], max_length=max_length,
                                        batch_size=batch_size)
            pos.update(a0 + p for p in ps[0])
        out.append(sorted(pos))
    return out

## Training loop (plain PyTorch — no HF Trainer dependency)

In [ ]:
#| export
def _pick_device(device: str = None):
    """`device` string or the best available: cuda → mps → cpu."""
    if device:
        return torch.device(device)
    if torch.cuda.is_available():
        return torch.device('cuda')
    if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


def _batch_order(rows: list[dict], batch_size: int) -> list[list[int]]:
    """Row indices as batches grouped by similar text length (char-count proxy).

    Similar-length rows keep per-batch padding tiny and, on MPS, cut the number
    of distinct tensor shapes the caching allocator must keep segments for.
    """
    order = sorted(range(len(rows)), key=lambda i: len(rows[i]['text']))
    return [order[i:i + batch_size] for i in range(0, len(rows), batch_size)]


def _autocast(device):
    """bf16 autocast context on cuda/mps; a no-op context elsewhere."""
    if device.type in ('cuda', 'mps'):
        return torch.autocast(device_type=device.type, dtype=torch.bfloat16)
    return contextlib.nullcontext()


def _progress(msg: str) -> None:
    """Append a timestamped `msg` line to `$TROUVER_PROGRESS_FILE`; no-op if unset."""
    path = os.environ.get('TROUVER_PROGRESS_FILE')
    if path:
        with open(path, 'a') as f:
            f.write(f"{time.strftime('%H:%M:%S')} {msg}\n")


def _eta(elapsed: float, step: int, nb: int, epochs: int, ep: int) -> str:
    """`'epoch_eta Xm total_eta Ym'` from the mean step speed so far."""
    sps = step / max(elapsed, 1e-9)
    return (f'epoch_eta {(nb - step) / sps / 60:.0f}m '
            f'total_eta {((epochs - ep) * nb + nb - step) / sps / 60:.0f}m')


def _epoch_eval_loss(model, rows: list[dict], tokenizer, device, max_length: int,
                     batch_size: int, cap: int = 256) -> float:
    """Mean loss over up to `cap` eval rows (no grad + autocast; train mode restored)."""
    model.eval()
    tot = n = 0
    with torch.no_grad():
        for i in range(0, min(cap, len(rows)), batch_size):
            mb = {k: v.to(device) for k, v in
                  division_feature_tensors(rows[i:i + batch_size], tokenizer,
                                           max_length).items()}
            with _autocast(device):
                tot += float(model(**mb).loss)
            n += 1
    model.train()
    return tot / max(n, 1)

def _epoch_report(model, tokenizer, eval_rows, device, max_length, batch_size,
                  ep: int, epochs: int, score: bool = False) -> None:
    """Epoch-end progress lines: `eval_loss`, plus seam P/R/F1 when `score`.

    `score_seam_f1` leaves the model in eval mode; restore train mode after it
    so mid-training scoring cannot silently disable dropout for later epochs.
    """
    _progress(f'epoch {ep}/{epochs} eval_loss '
              f'{_epoch_eval_loss(model, eval_rows, tokenizer, device, max_length, batch_size):.4f}')
    if score:
        m = score_seam_f1(model, tokenizer, eval_rows, max_length=max_length, batch_size=batch_size)
        _progress(f'epoch {ep}/{epochs} seam P {m["precision"]:.4f} R {m["recall"]:.4f} '
                  f'F1 {m["f1"]:.4f}')
        model.train()


def _fine_tune(model, rows: list[dict], tokenizer, device, epochs: int,
               batch_size: int, lr: float, max_length: int,
               eval_rows: list[dict] = None, out_dir: PathLike = None,
               score_every_epoch: bool = False) -> None:
    """AdamW over length-bucketed batches; bf16 autocast off-CPU + MPS cache trims.

    Batch order is reshuffled per epoch from the global RNG (seeded by
    `train_token_classifier`). Variable-length rows fragment the MPS pool until
    it OOMs, so every 50 steps on MPS we hand cached segments back.

    Progress sidecar (`$TROUVER_PROGRESS_FILE`, no-op when unset): a settings
    line at start, one line per 50 steps with loss + eta clocks, and an
    `eval_loss` line (plus a seam P/R/F1 line when `score_every_epoch`) after each
    epoch when `eval_rows` is given. With
    `out_dir`, model + tokenizer are also checkpointed to
    `out_dir/checkpoints/epoch-N` at every epoch boundary (crash insurance).
    """
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    _progress(f'start rows={len(rows)} epochs={epochs} batch_size={batch_size} '
              f'max_length={max_length} lr={lr} device={device.type}')
    t0 = time.monotonic()
    for ep in range(1, epochs + 1):
        batches = _batch_order(rows, batch_size)
        for step, bi in enumerate(torch.randperm(len(batches)).tolist(), 1):
            mb = division_feature_tensors([rows[k] for k in batches[bi]],
                                          tokenizer, max_length)
            mb = {k: v.to(device) for k, v in mb.items()}
            with _autocast(device):
                loss = model(**mb).loss
            opt.zero_grad(); loss.backward(); opt.step()
            if device.type == 'mps' and step % 50 == 0:
                torch.mps.empty_cache()
            if step % 50 == 0:
                _progress(f'epoch {ep}/{epochs} step {step}/{len(batches)} '
                          f'loss {loss.item():.4f} '
                          + _eta(time.monotonic() - t0, step, len(batches), epochs, ep))
        if eval_rows:
            _epoch_report(model, tokenizer, eval_rows, device, max_length, batch_size,
                          ep, epochs, score_every_epoch)
        if out_dir: _save_checkpoint(model, tokenizer, out_dir, ep)


def _save_checkpoint(model, tokenizer, out_dir: PathLike, epoch: int) -> None:
    """Save model + tokenizer to `out_dir/checkpoints/epoch-N` and log a line."""
    d = Path(out_dir) / 'checkpoints' / f'epoch-{epoch}'
    model.save_pretrained(d); tokenizer.save_pretrained(d)
    _progress(f'saved checkpoint {d}')


def train_token_classifier(rows: list[dict], model_id: str, out_dir: PathLike,
                           eval_rows: list[dict] = None, epochs: int = 3,
                           batch_size: int = 8, lr: float = 2e-5, max_length: int = 512,
                           device: str = None, seed: int = 0,
                           gradient_checkpointing: bool = False,
                           score_every_epoch: bool = False) -> dict:
    """
    Fine-tune `model_id` + 2-class head on division rows; save to `out_dir`; score eval.

    Labels are `{0: 'none', 1: 'split'}`. Rows are grouped into length-bucketed
    batches padded to their own longest row and trained under bf16 autocast
    off-CPU (with periodic MPS cache trims) — variable-length rows otherwise
    fragment the MPS pool until it OOMs well below the machine's real memory.
    Returns `score_seam_f1` on `eval_rows` (pure, by convention) or `None`.

    **Parameters**
    - `rows` (*list[dict]*) — Training rows (augmented view recommended; see notebook text).
    - `model_id` (*str*) — Encoder id, e.g. from `model_registry.get_model_id`.
    - `out_dir` (*PathLike*) — Directory for the saved HF model + tokenizer;
      per-epoch crash checkpoints land under its `checkpoints/` subdir.
    - `gradient_checkpointing` (*bool*) — Recompute activations in backward on
      cuda/mps (big memory saving, ~25-35% speed cost); no-op on cpu.
    - `score_every_epoch` (*bool*) — Also log seam P/R/F1 per epoch to the progress
      sidecar (needs `eval_rows`; one full decode pass over them each epoch).
    - `eval_rows` (*list[dict]*) — Optional rows scored after training (leakage-free
      by caller); also logged as epoch-end `eval_loss` in the progress sidecar.
    - `epochs` (*int*) / `batch_size` (*int*) / `lr` (*float*) — Standard knobs.
    - `max_length` (*int*) — Window cap; must match `score_seam_f1` at eval time.
    - `device` (*str*) — Force a device (`'cpu'`, `'mps'`, `'cuda'`); default = best available.
    - `seed` (*int*) — Torch seed for run reproducibility.
    """
    torch.manual_seed(seed)
    device = _pick_device(device)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForTokenClassification.from_pretrained(
        model_id, num_labels=2, id2label={0: 'none', 1: 'split'},
        label2id={'none': 0, 'split': 1}).to(device)
    if gradient_checkpointing and device.type in ('cuda', 'mps'):
        model.gradient_checkpointing_enable()
    _fine_tune(model, rows, tokenizer, device, epochs, batch_size, lr,
               max_length, eval_rows, out_dir, score_every_epoch)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    model.save_pretrained(out_dir); tokenizer.save_pretrained(out_dir)
    return None if eval_rows is None else score_seam_f1(
        model, tokenizer, eval_rows, max_length=max_length, batch_size=batch_size)


In [ ]:
#| hide
import tempfile
from pathlib import Path

from fastcore.test import *

from trouver.helper.tests import _test_directory
from trouver.machine_learning.ml_data import PROVENANCE_COLUMN, load_rows
from trouver.machine_learning.producers.note_division import NOTE_DIVISION_SCHEMA

FIXTURE = _test_directory() / 'ml' / 'note_division_demo.jsonl'
TINY = 'hf-internal-testing/tiny-random-BertForTokenClassification'


In [ ]:
# End-to-end on the synthetic fixture with a tiny random model: shapes, save, reload,
# and metric structure (tiny random weights — only structure is asserted, per spec).
rows = load_rows(FIXTURE, schema=NOTE_DIVISION_SCHEMA)
assert len(rows) == 3 and all(r[PROVENANCE_COLUMN].startswith("_index_demo_") for r in rows)
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(TINY)
feats = division_feature_tensors([dict(r) for r in rows], tok)
assert feats['input_ids'].shape == feats['labels'].shape
assert int((feats['labels'] == 1).sum()) == 3  # one seam per row survived alignment
with tempfile.TemporaryDirectory() as td:
    res = train_token_classifier([dict(r) for r in rows], TINY, td,
                                 eval_rows=[dict(r) for r in rows], epochs=1,
                                 batch_size=2, max_length=64, device='cpu')
    assert (Path(td) / 'config.json').exists()
    assert set(res) == {'precision', 'recall', 'f1'}
    assert all(0.0 <= v <= 1.0 for v in res.values())


In [ ]:
# length-bucketing + autocast helpers (MPS memory fix)
bs_ = _batch_order(rows, 3)
test_eq(sum(len(b) for b in bs_), len(rows))           # every row kept exactly once
test_eq(sorted(i for b in bs_ for i in b), list(range(len(rows))))
for b in bs_:                                          # each batch sorted by text length
    ls = [len(rows[i]['text']) for i in b]
    test_eq(ls, sorted(ls))
test_eq(len(bs_), -(-len(rows) // 3))                  # ceil division
import contextlib
assert isinstance(_autocast(torch.device('cpu')), contextlib.nullcontext)
with _autocast(torch.device('cpu')):                   # usable as a context manager
    pass

In [ ]:
# Characterization net for `score_seam_f1` (pins behavior before the predict refactor):
# a constant-logit model must yield all-zero metrics (class 0 everywhere) and full
# recall (class 1 everywhere — every token end is predicted, so every gold seam,
# which is some token's end offset, is covered). Placed after `_autocast` exists.
import types as _types

from fastcore.test import test_eq
from transformers import AutoTokenizer

class _ConstModel(torch.nn.Module):
    """Fake token classifier always emitting `cls` — deterministic scoring net."""
    def __init__(self, cls: int):
        super().__init__()
        self.cls = cls
        self.device = torch.device('cpu')
    def forward(self, input_ids=None, attention_mask=None):
        z = torch.zeros(*input_ids.shape, 2)
        z[..., self.cls] = 1.0
        return _types.SimpleNamespace(logits=z)

_crows = [{'text': 'ab cd', 'spans': [{'start': 2, 'end': 2, 'label': 'split'}]},
          {'text': 'xy', 'spans': []}]
_ctok = AutoTokenizer.from_pretrained(
    'hf-internal-testing/tiny-random-BertForTokenClassification')
test_eq(score_seam_f1(_ConstModel(0), _ctok, _crows, max_length=64, batch_size=2),
        {'precision': 0.0, 'recall': 0.0, 'f1': 0.0})
_m1 = score_seam_f1(_ConstModel(1), _ctok, _crows, max_length=64, batch_size=2)
test_eq(_m1['recall'], 1.0)
assert 0.0 < _m1['precision'] <= 1.0 and _m1['f1'] > 0.0


In [ ]:
# predict_seam_positions: all-ones model predicts every non-blank token end;
# all-zeros predicts nothing; scores and positions agree through one rule.
pred = predict_seam_positions(_ConstModel(1), _ctok, [r['text'] for r in _crows],
                              max_length=64, batch_size=2)
offs0 = _ctok('ab cd', add_special_tokens=False,
              return_offsets_mapping=True)['offset_mapping']
test_eq(pred[0], sorted({e for b, e in offs0 if e > b}))  # exact position set, sorted
assert 2 in pred[0]                                       # the gold seam is among them
test_eq(predict_seam_positions(_ConstModel(0), _ctok, ['ab cd'], max_length=64), [[]])

# Batch-split invariance: one batch vs per-row batches give identical positions.
test_eq(pred, predict_seam_positions(_ConstModel(1), _ctok,
                                     [r['text'] for r in _crows], max_length=64, batch_size=1))

# The refactor invariant: score_seam_f1 == metrics recomputed from the positions.
tp_ = sum(len(set(p) & {s['start'] for s in r['spans'] if s['label'] == 'split'})
          for r, p in zip(_crows, pred))
fp_ = sum(len(set(p) - {s['start'] for s in r['spans'] if s['label'] == 'split'})
          for r, p in zip(_crows, pred))
fn_ = sum(len({s['start'] for s in r['spans'] if s['label'] == 'split'} - set(p))
          for r, p in zip(_crows, pred))
ep, er = tp_ / (tp_ + fp_), tp_ / (tp_ + fn_)
test_eq(score_seam_f1(_ConstModel(1), _ctok, _crows, max_length=64, batch_size=2),
        {'precision': ep, 'recall': er, 'f1': 2 * ep * er / (ep + er)})


In [ ]:
# predict_seam_positions_windowed: identical to predict_seam_positions on short texts;
# on an over-cap text it loses nothing (all-ones model -> every token end in the FULL text).
# Placed after `_autocast` exists, like the `score_seam_f1` net above.
import types as _types

import torch
from fastcore.test import test_eq
from transformers import AutoTokenizer

class _WConstModel(torch.nn.Module):
    """Fake token classifier always emitting `cls` — deterministic windowed net."""
    def __init__(self, cls: int):
        super().__init__()
        self.cls = cls
        self.device = torch.device('cpu')
    def forward(self, input_ids=None, attention_mask=None):
        z = torch.zeros(*input_ids.shape, 2)
        z[..., self.cls] = 1.0
        return _types.SimpleNamespace(logits=z)

_wtok = AutoTokenizer.from_pretrained('hf-internal-testing/tiny-random-BertForTokenClassification')
_short = 'ab cd'
test_eq(predict_seam_positions_windowed(_WConstModel(1), _wtok, [_short], max_length=64),
        predict_seam_positions(_WConstModel(1), _wtok, [_short], max_length=64))
test_eq(predict_seam_positions_windowed(_WConstModel(0), _wtok, ['note one is here ' * 40],
                                        max_length=16, overlap=4), [[]])

_long = 'note one is here ' * 40
_all_ends = sorted({e for b, e in _wtok(_long, add_special_tokens=False,
                                         return_offsets_mapping=True)['offset_mapping'] if e > b})
_w = predict_seam_positions_windowed(_WConstModel(1), _wtok, [_long], max_length=16, overlap=4)
test_eq(_w[0], _all_ends)                                    # nothing truncated away
# vs plain (truncating) predict, which only sees the first window.
assert set(predict_seam_positions(_WConstModel(1), _wtok, [_long], max_length=16)[0]) < set(_all_ends)

In [ ]:
# _progress sidecar logging ($TROUVER_PROGRESS_FILE)
import os as _os, tempfile as _tf
_p = Path(_tf.mkdtemp()) / 'prog.log'
_os.environ['TROUVER_PROGRESS_FILE'] = str(_p)
_progress('hello'); _progress('world')
lines = _p.read_text().splitlines()
test_eq([l.split(' ', 1)[1] for l in lines], ['hello', 'world'])  # ts prefix + msg
del _os.environ['TROUVER_PROGRESS_FILE']
_progress('dropped')                                              # no-op when unset
test_eq(len(_p.read_text().splitlines()), 2)

In [ ]:
# eta clocks + epoch eval loss (progress v2)
test_eq(_eta(100.0, 100, 1000, 3, 1), 'epoch_eta 15m total_eta 48m')
test_eq(_eta(50.0, 100, 100, 1, 1), 'epoch_eta 0m total_eta 0m')   # last batch done
_m = AutoModelForTokenClassification.from_pretrained(TINY, num_labels=2)
_v = _epoch_eval_loss(_m, [dict(r) for r in rows], tok, torch.device('cpu'), 64, 2)
assert isinstance(_v, float) and _v > 0                             # tiny random model: finite positive mean
test_eq(_epoch_eval_loss(_m, [], tok, torch.device('cpu'), 64, 2), 0.0)

In [ ]:
# per-epoch crash checkpoints + gradient-checkpointing flag (v3)
import tempfile
with tempfile.TemporaryDirectory() as td_:
    _save_checkpoint(_m, tok, td_, 1)
    assert (Path(td_) / 'checkpoints' / 'epoch-1' / 'config.json').exists()
    res_ = train_token_classifier([dict(r) for r in rows], TINY, td_,
                                  eval_rows=[dict(r) for r in rows], epochs=1,
                                  batch_size=2, max_length=64, device='cpu',
                                  gradient_checkpointing=True)   # no-op on cpu path
    assert set(res_) == {'precision', 'recall', 'f1'}
    assert (Path(td_) / 'checkpoints' / 'epoch-1').exists()      # epoch save ran

In [ ]:
# score_every_epoch: seam P/R/F1 lands in the progress sidecar at each epoch end.
import tempfile
with tempfile.TemporaryDirectory() as td2, tempfile.TemporaryDirectory() as pf:
    _prog = Path(pf) / 'progress.log'
    os.environ['TROUVER_PROGRESS_FILE'] = str(_prog)
    try:
        train_token_classifier([dict(r) for r in rows], TINY, td2,
                               eval_rows=[dict(r) for r in rows], epochs=1,
                               batch_size=2, max_length=64, device='cpu',
                               score_every_epoch=True)
    finally:
        del os.environ['TROUVER_PROGRESS_FILE']
    _log = _prog.read_text()
    assert 'eval_loss' in _log and 'seam P' in _log and 'F1' in _log